# Multi-dataset validation summaries

This notebook compares successful batch jobs using the fixed IPv4 overall and native selected-prefix scope. Failed and pending jobs remain visible in provenance tables but are excluded from aggregation. Absolute difference is defined as `median_of_prefix_medians - overall_median`; no ratio or scientific conclusion is inferred here.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from mawi_global_analysis.comparison import build_comparison_flow_inclusion
from mawi_global_analysis.io import load_batch

batch_manifest_value = os.environ.get('MAWI_BATCH_MANIFEST')
if not batch_manifest_value:
    raise ValueError('set MAWI_BATCH_MANIFEST to a batch_manifest.json path')

batch = load_batch(Path(batch_manifest_value))


In [ ]:
job_status_table = pd.DataFrame([
    {
        'dataset_id': job.dataset_id,
        'config_path': job.config_path,
        'config_hash': job.config_hash,
        'status': job.status,
        'linked_run_manifest': str(job.linked_run_manifest) if job.linked_run_manifest else None,
    }
    for job in batch.jobs
])

batch_provenance_table = pd.DataFrame([
    {
        'batch_name': batch.batch_name,
        'batch_status': batch.status,
        'dataset_list_path': batch.dataset_list.get('path'),
        'dataset_list_hash': batch.dataset_list.get('hash'),
        'ordered_dataset_count': len(batch.dataset_list.get('dataset_ids', [])),
        'config_count': len(batch.configs),
        'total_jobs': len(batch.jobs),
        'succeeded_jobs': int((job_status_table['status'] == 'succeeded').sum()),
        'failed_jobs': int((job_status_table['status'] == 'failed').sum()),
        'pending_jobs': int((job_status_table['status'] == 'pending').sum()),
    }
])

display(batch_provenance_table)
display(job_status_table)


In [ ]:
CONDITIONS = (('Raw', 'raw_included'), ('Strict', 'strict_included'), ('Broad', 'broad_included'))
COUNT_COLUMNS = ('flow_count', 'packet_count', 'frame_byte_count')
MEDIAN_METRICS = ('median_packet_count', 'median_frame_byte_count', 'median_duration')


def condition_metrics(frame, condition, inclusion_column):
    survivors = frame.loc[frame[inclusion_column]]
    return {
        'condition': condition,
        'flow_count': len(survivors),
        'packet_count': survivors['packet_count'].sum(),
        'frame_byte_count': survivors['frame_byte_count'].sum(),
        'median_packet_count': survivors['packet_count'].median(),
        'median_frame_byte_count': survivors['frame_byte_count'].median(),
        'median_duration': survivors['duration'].median(),
    }


def raw_relative_ratio(removed, raw_total):
    return removed / raw_total if raw_total else np.nan


def prefix_metrics(survivors):
    if survivors.empty:
        return pd.DataFrame(columns=('analysis_prefix', *COUNT_COLUMNS, *MEDIAN_METRICS))
    return survivors.groupby('analysis_prefix', as_index=False).agg(
        flow_count=('flow_id', 'size'),
        packet_count=('packet_count', 'sum'),
        frame_byte_count=('frame_byte_count', 'sum'),
        median_packet_count=('packet_count', 'median'),
        median_frame_byte_count=('frame_byte_count', 'median'),
        median_duration=('duration', 'median'),
    )


dataset_condition_rows = []
dataset_removal_rows = []
prefix_condition_tables = []
prefix_distribution_rows = []
overall_vs_prefix_rows = []
sanity_rows = []

for job in batch.successful_jobs:
    run = batch.load_job_run(job)
    inclusion = build_comparison_flow_inclusion(run.flows, run.labels)
    flow_metrics = run.flows.merge(inclusion, on='flow_id', how='inner', validate='one_to_one')
    ipv4_flows = flow_metrics.loc[pd.to_numeric(flow_metrics['ip_version']) == 4].copy()

    selected_prefixes = run.prefixes.loc[
        run.prefixes['selected_for_analysis'] == True, ['prefix']
    ].rename(columns={'prefix': 'analysis_prefix'}).copy()
    if selected_prefixes['analysis_prefix'].duplicated().any():
        raise ValueError(f'duplicate selected native prefix for {job.dataset_id}/{job.config_hash}')
    selected_prefix_ids = set(selected_prefixes['analysis_prefix'])

    native_membership = run.membership.loc[run.membership['analysis_scope'] == 'native'].copy()
    if not set(native_membership['flow_id']).issubset(set(inclusion['flow_id'])):
        raise ValueError(f'native membership has an unknown flow_id for {job.dataset_id}/{job.config_hash}')
    if not set(native_membership['analysis_prefix']).issubset(selected_prefix_ids):
        raise ValueError(f'native membership changed the selected prefix set for {job.dataset_id}/{job.config_hash}')

    prefix_flows = native_membership.merge(
        ipv4_flows.loc[:, ['flow_id', 'packet_count', 'frame_byte_count', 'duration', 'raw_included', 'strict_included', 'broad_included']],
        on='flow_id',
        how='inner',
        validate='many_to_one',
    )

    metadata = {'dataset_id': job.dataset_id, 'config_path': job.config_path, 'config_hash': job.config_hash}
    per_condition = pd.DataFrame([condition_metrics(ipv4_flows, *condition) for condition in CONDITIONS])
    per_condition = pd.concat([pd.DataFrame([metadata] * len(per_condition)), per_condition], axis=1)
    per_condition['selected_native_prefix_count'] = len(selected_prefixes)
    dataset_condition_rows.extend(per_condition.to_dict('records'))

    raw_condition = per_condition.loc[per_condition['condition'] == 'Raw'].iloc[0]
    for condition in ('Strict', 'Broad'):
        survivor = per_condition.loc[per_condition['condition'] == condition].iloc[0]
        removed_flow_count = raw_condition.flow_count - survivor.flow_count
        removed_packet_count = raw_condition.packet_count - survivor.packet_count
        removed_frame_byte_count = raw_condition.frame_byte_count - survivor.frame_byte_count
        dataset_removal_rows.append({
            **metadata,
            'condition': condition,
            'removed_flow_count': removed_flow_count,
            'removed_flow_ratio': raw_relative_ratio(removed_flow_count, raw_condition.flow_count),
            'removed_packet_count': removed_packet_count,
            'removed_packet_ratio': raw_relative_ratio(removed_packet_count, raw_condition.packet_count),
            'removed_frame_byte_count': removed_frame_byte_count,
            'removed_frame_byte_ratio': raw_relative_ratio(removed_frame_byte_count, raw_condition.frame_byte_count),
        })

    prefix_condition_index = selected_prefixes.merge(
        pd.DataFrame(CONDITIONS, columns=('condition', 'inclusion_column')), how='cross'
    )
    prefix_metric_frames = []
    for condition, inclusion_column in CONDITIONS:
        metrics = prefix_metrics(prefix_flows.loc[prefix_flows[inclusion_column]]).copy()
        metrics['condition'] = condition
        prefix_metric_frames.append(metrics)
    prefix_condition = prefix_condition_index.drop(columns='inclusion_column').merge(
        pd.concat(prefix_metric_frames, ignore_index=True),
        on=('analysis_prefix', 'condition'),
        how='left',
        validate='one_to_one',
    )
    for column in COUNT_COLUMNS:
        prefix_condition[column] = prefix_condition[column].fillna(0)
    prefix_condition = pd.concat([pd.DataFrame([metadata] * len(prefix_condition)), prefix_condition], axis=1)

    overall_counts = per_condition.set_index('condition')['flow_count']
    if not (overall_counts['Broad'] <= overall_counts['Strict'] <= overall_counts['Raw']):
        raise ValueError(f'overall survivors violate Broad <= Strict <= Raw for {job.dataset_id}/{job.config_hash}')
    for condition, _ in CONDITIONS:
        condition_prefixes = set(prefix_condition.loc[prefix_condition['condition'] == condition, 'analysis_prefix'])
        if condition_prefixes != selected_prefix_ids:
            raise ValueError(f'prefix identity changed for {condition} in {job.dataset_id}/{job.config_hash}')
        if (prefix_condition['condition'] == condition).sum() != len(selected_prefixes):
            raise ValueError(f'prefix count changed for {condition} in {job.dataset_id}/{job.config_hash}')

    prefix_condition_tables.append(prefix_condition)
    for condition, _ in CONDITIONS:
        condition_prefix_summary = prefix_condition.loc[prefix_condition['condition'] == condition]
        overall_summary = per_condition.loc[per_condition['condition'] == condition].iloc[0]
        for metric in MEDIAN_METRICS:
            values = condition_prefix_summary[metric].dropna()
            prefix_distribution_rows.append({
                **metadata, 'condition': condition, 'metric': metric,
                'valid_prefix_count': len(values), 'q25': values.quantile(0.25),
                'median': values.median(), 'q75': values.quantile(0.75),
            })
            prefix_median = values.median()
            overall_vs_prefix_rows.append({
                **metadata, 'condition': condition, 'metric': metric,
                'overall_median': overall_summary[metric],
                'median_of_prefix_medians': prefix_median,
                'absolute_difference': prefix_median - overall_summary[metric],
            })

    sanity_rows.append({
        **metadata,
        'selected_prefix_count': len(selected_prefixes),
        'raw_flow_count': overall_counts['Raw'],
        'strict_flow_count': overall_counts['Strict'],
        'broad_flow_count': overall_counts['Broad'],
        'strict_zero_survivor_prefixes': int(((prefix_condition['condition'] == 'Strict') & (prefix_condition['flow_count'] == 0)).sum()),
        'broad_zero_survivor_prefixes': int(((prefix_condition['condition'] == 'Broad') & (prefix_condition['flow_count'] == 0)).sum()),
    })

    del run, inclusion, flow_metrics, ipv4_flows, native_membership, prefix_flows, prefix_condition

dataset_condition_summary = pd.DataFrame(dataset_condition_rows)
dataset_removal_summary = pd.DataFrame(dataset_removal_rows)
prefix_condition_summary = pd.concat(prefix_condition_tables, ignore_index=True) if prefix_condition_tables else pd.DataFrame()
prefix_distribution_summary = pd.DataFrame(prefix_distribution_rows)
overall_vs_prefix_summary = pd.DataFrame(overall_vs_prefix_rows)
cross_dataset_sanity_summary = pd.DataFrame(sanity_rows)


In [ ]:
display(dataset_condition_summary)
display(dataset_removal_summary)
display(prefix_condition_summary)
display(prefix_distribution_summary)
display(overall_vs_prefix_summary)
display(cross_dataset_sanity_summary)
